**Ερώτημα 2ο**

Import Dataset & Normalization

In [ ]:
import tensorflow as tf

# Load the Fashion-MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalization values of pixel (0-1)
x_train, x_test = x_train / 255.0, x_test / 255.0

print(f"Train samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")
print(f"Image shape: {x_train[0].shape}")

Δημιουργία, Εκπαίδευση & Αξιολόγηση Μοντέλων

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras import models, layers, backend as K

# --- Create Models ---

def create_dnn():
    model = models.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax')
    ], name='DNN_from_scratch')

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def create_cnn():
    model = models.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Reshape((28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name='CNN_from_scratch')

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Helping function for metrics calculation
def get_metrics(y_true, y_pred_probs):
    y_pred = np.argmax(y_pred_probs, axis=1)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return acc, prec, rec, f1

# --- Train & Evaluate w/ K-Fold ---

EPOCHS = 150
BATCH_SIZE = 256
N_SPLITS = 6

kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# List to store results
results_list = []

for fold_no, (train_idx, val_idx) in enumerate(kfold.split(x_train, y_train), 1):
    print(f'\n--- Fold {fold_no}/{N_SPLITS} ---')

    x_tr_fold, y_tr_fold = x_train[train_idx], y_train[train_idx]
    x_val_fold, y_val_fold = x_train[val_idx], y_train[val_idx]

    # --- Train DNN ---
    print("Εκπαίδευση DNN_from_scratch...")
    dnn = create_dnn()
    dnn.fit(x_tr_fold, y_tr_fold,
            epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
            validation_data=(x_val_fold, y_val_fold))

    # Evaluation DNN
    dnn_val_preds = dnn.predict(x_val_fold, verbose=0)
    dnn_test_preds = dnn.predict(x_test, verbose=0)

    dnn_val_metrics = get_metrics(y_val_fold, dnn_val_preds)
    dnn_test_metrics = get_metrics(y_test, dnn_test_preds)

    # Save to list
    results_list.append(['DNN_from_scratch', 'Train', fold_no] + list(dnn_val_metrics))
    results_list.append(['DNN_from_scratch', 'Test', fold_no] + list(dnn_test_metrics))

    # --- Train CNN ---
    print("Εκπαίδευση CNN_from_scratch...")
    cnn = create_cnn()
    cnn.fit(x_tr_fold, y_tr_fold,
            epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
            validation_data=(x_val_fold, y_val_fold))

    # Evaluation CNN
    cnn_val_preds = cnn.predict(x_val_fold, verbose=0)
    cnn_test_preds = cnn.predict(x_test, verbose=0)

    cnn_val_metrics = get_metrics(y_val_fold, cnn_val_preds)
    cnn_test_metrics = get_metrics(y_test, cnn_test_preds)

    # Save to list
    results_list.append(['CNN_from_scratch', 'Train', fold_no] + list(cnn_val_metrics))
    results_list.append(['CNN_from_scratch', 'Test', fold_no] + list(cnn_test_metrics))

    # Release GPU
    K.clear_session()

print("\nΗ εκπαίδευση 150 epochs για τα 6 Folds ολοκληρώθηκε")

# Print summaries using our functions
print("--- DNN Architecture ---")
temp_dnn = create_dnn()
temp_dnn.summary()

print("\n--- CNN Architecture ---")
temp_cnn = create_cnn()
temp_cnn.summary()

Transfer Learning & Fine-Tuning

In [ ]:
import pandas as pd
from tensorflow.keras.models import load_model

# --- Transfer Learning & Fine-Tuning ---

print("\n=== Έναρξη Transfer Learning ===")

DNN_MODEL_PATH = 'best_dnn_model.h5'
CNN_MODEL_PATH = 'best_cnn_model.h5'

for fold_no, (train_idx, val_idx) in enumerate(kfold.split(x_train, y_train), 1):
    print(f'\n--- Transfer Learning Fold {fold_no}/{N_SPLITS} ---')

    x_tr_fold, y_tr_fold = x_train[train_idx], y_train[train_idx]
    x_val_fold, y_val_fold = x_train[val_idx], y_train[val_idx]

    # ==========================================
    # DNN Transfer
    # ==========================================
    print("Φόρτωση και εκπαίδευση DNN_transfer...")

    # Load fold loop
    dnn_transfer = load_model(DNN_MODEL_PATH, compile=False)
    dnn_transfer._name = 'DNN_transfer'

    # Freezing classification layer
    for layer in dnn_transfer.layers[:-1]:
        layer.trainable = False

    dnn_transfer.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Train ONLY last layer for 40 epochs
    dnn_transfer.fit(x_tr_fold, y_tr_fold, epochs=40, batch_size=BATCH_SIZE, verbose=0)

    # Unfreezing (Fine-tuning)
    for layer in dnn_transfer.layers:
        layer.trainable = True

    dnn_transfer.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Train whole architecture end-to-end for 20 epochs
    dnn_transfer.fit(x_tr_fold, y_tr_fold, epochs=20, batch_size=BATCH_SIZE, verbose=0)

    # Evaluate & Save
    dnn_t_val_preds = dnn_transfer.predict(x_val_fold, verbose=0)
    dnn_t_test_preds = dnn_transfer.predict(x_test, verbose=0)

    dnn_t_val_metrics = get_metrics(y_val_fold, dnn_t_val_preds)
    dnn_t_test_metrics = get_metrics(y_test, dnn_t_test_preds)

    results_list.append(['DNN_transfer', 'Train', fold_no] + list(dnn_t_val_metrics))
    results_list.append(['DNN_transfer', 'Test', fold_no] + list(dnn_t_test_metrics))


    # ==========================================
    # CNN Transfer
    # ==========================================
    print("Φόρτωση και εκπαίδευση CNN_transfer...")

    cnn_transfer = load_model(CNN_MODEL_PATH, compile=False)
    cnn_transfer._name = 'CNN_transfer'

    # Freezing classification layer
    for layer in cnn_transfer.layers[:-1]:
        layer.trainable = False

    cnn_transfer.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Train ONLY last layer for 40 epochs
    cnn_transfer.fit(x_tr_fold, y_tr_fold, epochs=40, batch_size=BATCH_SIZE, verbose=0)

    # Unfreezing (Fine-tuning)
    for layer in cnn_transfer.layers:
        layer.trainable = True

    cnn_transfer.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Train whole architecture end-to-end for 20 epochs
    cnn_transfer.fit(x_tr_fold, y_tr_fold, epochs=20, batch_size=BATCH_SIZE, verbose=0)

    # Evaluate & Save
    cnn_t_val_preds = cnn_transfer.predict(x_val_fold, verbose=0)
    cnn_t_test_preds = cnn_transfer.predict(x_test, verbose=0)

    cnn_t_val_metrics = get_metrics(y_val_fold, cnn_t_val_preds)
    cnn_t_test_metrics = get_metrics(y_test, cnn_t_test_preds)

    results_list.append(['CNN_transfer', 'Train', fold_no] + list(cnn_t_val_metrics))
    results_list.append(['CNN_transfer', 'Test', fold_no] + list(cnn_test_metrics))

    # Release GPU
    K.clear_session()

print("\nΗ εκπαίδευση Transfer Learning ολοκληρώθηκε")

# --- Create DataFrame and save in CSV ---

# Defining column names as requested
columns = [
    'Technique name',
    'Set',
    'Fold number',
    'Accuracy',
    'Precision',
    'Recall',
    'F1 score'
]

# Create Pandas DataFrame from list
df_results = pd.DataFrame(results_list, columns=columns)

# Save to file
CSV_FILENAME = 'erotima2.csv'
df_results.to_csv(CSV_FILENAME, index=False)

print(f"\nΤο DataFrame δημιουργήθηκε με {len(df_results)} γραμμές.")
print(f"Τα αποτελέσματα αποθηκεύτηκαν στο αρχείο: {CSV_FILENAME}")

# Show the first and last lines for confirmation
display(df_results)

Confusion Matrices & ROC Curves

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize

# 1. Confusion Matrices
models_to_plot = {
    'DNN_from_scratch': dnn,
    'CNN_from_scratch': cnn,
    'DNN_transfer': dnn_transfer,
    'CNN_transfer': cnn_transfer
}

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, (name, model) in enumerate(models_to_plot.items()):
    preds = np.argmax(model.predict(x_test, verbose=0), axis=1)
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=axes[i], cmap='Blues', values_format='d')
    axes[i].set_title(f'Confusion Matrix: {name}')

plt.tight_layout()
plt.show()

# 2. ROC Curves - Macro-average for Multi-class
plt.figure(figsize=(10, 8))

# Binarize labels for ROC calculation
y_test_bin = label_binarize(y_test, classes=np.arange(10))

for name, model in models_to_plot.items():
    y_score = model.predict(x_test, verbose=0)

    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    # For simplicity, calculate the macro-average ROC
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    plt.plot(fpr["micro"], tpr["micro"],
             label=f'{name} (area = {roc_auc["micro"]:0.2f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (Micro-Averaged)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

Boxplots, Bar Charts & Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# 1. Boxplots for Accuracy
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_results, x='Technique name', y='Accuracy', hue='Set')
plt.title('Boxplot of Accuracy per Technique and Set')
plt.show()

# 2. Bar Charts with Error Bars
plt.figure(figsize=(10, 6))
sns.barplot(data=df_results, x='Technique name', y='Accuracy', hue='Set', capsize=.1, errorbar='sd')
plt.title('Mean Accuracy with Standard Deviation Bars')
plt.ylim(0.8, 1.0)
plt.show()

# 3. Heatmap - Test Set
df_heatmap = df_results[df_results['Set'] == 'Test'].groupby('Technique name')[['Accuracy', 'Precision', 'Recall', 'F1 score']].mean()
plt.figure(figsize=(10, 5))
sns.heatmap(df_heatmap, annot=True, cmap='YlGnBu', fmt='.4f')
plt.title('Heatmap: Average Metrics on Test Set')
plt.show()